# 13 Beta Diversity and symptom-related gluten avoidance

We test whether fungal community composition differs between participants who report avoiding gluten because it makes them feel bad and those who report no gluten-related issue (`gluten_sample`). We compare whole-ASV and 16-mer Bray–Curtis distances using the same samples for both representations and both models. The adjusted model tests gluten consumption patterns after accounting for age and sex.

## 1. Setup

### 1.1 Imports

In [1]:
import subprocess
from pathlib import Path

import pandas as pd
import qiime2 as q2
from skbio import DistanceMatrix
from qiime2 import Visualization, Artifact, Metadata

### 1.2 Notebook settings and paths

We define the input paths, the shared complete-case files, and the outputs for each feature representation and model.

In [5]:
# 1. Shared inputs and output directories
metadata_path = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"

out_dir = "../data/processed_ignored/13_beta_diversity_gluten"
vis_dir = "../data/processed/13_beta_diversity_gluten"


# 2. Complete-case inputs for both models
adjusted_gluten_out_dir = f"{out_dir}/adjusted_gluten"

asv_gluten_distance_path = f"{adjusted_gluten_out_dir}/asv_bray_curtis_complete.qza"
kmer16_gluten_distance_path = f"{adjusted_gluten_out_dir}/kmer16_bray_curtis_complete.qza"

gluten_metadata_path = f"{adjusted_gluten_out_dir}/metadata_complete.tsv"


# 3. Overall PERMANOVA outputs
unadjusted_permanova_dir = f"{vis_dir}/unadjusted_permanova"
adjusted_permanova_dir = f"{vis_dir}/adjusted_permanova"

asv_unadjusted_overall_path = f"{unadjusted_permanova_dir}/asv_overall.qzv"
kmer16_unadjusted_overall_path = f"{unadjusted_permanova_dir}/kmer16_overall.qzv"

asv_adjusted_overall_path = f"{adjusted_permanova_dir}/asv_overall.qzv"
kmer16_adjusted_overall_path = f"{adjusted_permanova_dir}/kmer16_overall.qzv"


# Create directories
!mkdir -p "$out_dir" "$vis_dir" "$adjusted_gluten_out_dir"
!mkdir -p "$unadjusted_permanova_dir" "$adjusted_permanova_dir"

### 1.3 Load inputs

We load the whole-ASV and 16-mer Bray–Curtis distance matrices from notebook 07 and the cleaned metadata from notebook 06.

In [4]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
kmer16_distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)
asv_distance_matrix = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)

## 2. Select samples

### 2.1 Match sample IDs

We keep samples present in both Bray–Curtis matrices and in the metadata.

In [21]:
columns = ["gluten_sample", "age_years_sample", "sex_sample"]

unfiltered_gluten_metadata = metadata[columns].dropna()

unfiltered_gluten_metadata.index.name = "sample-id"

print("Complete samples:", len(unfiltered_gluten_metadata))
display(unfiltered_gluten_metadata["gluten_sample"].value_counts())

Complete samples: 141


gluten_sample
No                                                                               110
I do not eat gluten because it makes me feel bad                                  26
I was diagnosed with gluten allergy (anti-gluten IgG), but not celiac disease      4
I was diagnosed with celiac disease                                                1
Name: count, dtype: int64

### 2.2 Keep complete cases

We keep samples labeled with sex, numeric age, settlement type, and recorded diet type. Both PERMANOVA models use this same subset.

In [22]:
gluten_groups = [
    "No",
    "I do not eat gluten because it makes me feel bad",
]

gluten_metadata = unfiltered_gluten_metadata.loc[
    unfiltered_gluten_metadata["gluten_sample"].isin(gluten_groups)
].copy()

gluten_metadata.index.name = "sample-id"

q2.Metadata(gluten_metadata).save(f"{gluten_metadata_path}")

print("Complete-case samples:", len(gluten_metadata))
display(gluten_metadata["gluten_sample"].value_counts())

Complete-case samples: 136


gluten_sample
No                                                  110
I do not eat gluten because it makes me feel bad     26
Name: count, dtype: int64

In [23]:
gluten_metadata.head()

,gluten_sample,age_years_sample,sex_sample
sample-id,,,
ERR5327198,No,67.0,female
ERR5327266,I do not eat gluten because it makes me feel bad,28.0,female
ERR5327282,No,26.0,female
ERR5327284,No,25.0,female
ERR5327285,No,27.0,male


In [25]:
!qiime diversity filter-distance-matrix \
    --i-distance-matrix "$asv_distance_path" \
    --m-metadata-file "$gluten_metadata_path" \
    --o-filtered-distance-matrix "$asv_gluten_distance_path"

!qiime diversity filter-distance-matrix \
    --i-distance-matrix "$kmer16_distance_path" \
    --m-metadata-file "$gluten_metadata_path" \
    --o-filtered-distance-matrix "$kmer16_gluten_distance_path"

Saved DistanceMatrix to: ../data/processed_ignored/13_beta_diversity_gluten/adjusted_gluten/asv_bray_curtis_complete.qza
Saved DistanceMatrix to: ../data/processed_ignored/13_beta_diversity_gluten/adjusted_gluten/kmer16_bray_curtis_complete.qza


### 3.1 Unadjusted PERMANOVA

We test whether fungal community composition differs across BMI values. Both tests use the same 136 samples and include gluten consumption as the only predictor.

#### 3.1.1 Whole ASVs

In [26]:
!qiime diversity adonis \
    --i-distance-matrix "$asv_gluten_distance_path" \
    --m-metadata-file "$gluten_metadata_path" \
    --p-formula "gluten_sample" \
    --o-visualization "$asv_unadjusted_overall_path"

Saved Visualization to: ../data/processed/13_beta_diversity_gluten/unadjusted_permanova/asv_overall.qzv


In [27]:
Visualization.load(asv_unadjusted_overall_path)

<visualization: Visualization uuid: a25cd935-feeb-4bb4-83d9-e78f54eda4da>

#### 3.1.2 16-mers

In [28]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_gluten_distance_path" \
    --m-metadata-file "$gluten_metadata_path" \
    --p-formula "gluten_sample" \
    --o-visualization "$kmer16_unadjusted_overall_path"

Saved Visualization to: ../data/processed/13_beta_diversity_gluten/unadjusted_permanova/kmer16_overall.qzv


In [29]:
Visualization.load(kmer16_unadjusted_overall_path)

<visualization: Visualization uuid: d43c6299-a137-459e-a4ac-d9aa754d4027>

#### Unadjusted results

| **Gluten term** | **Whole ASVs** | **16-mers** |
| --------------- | -------------: | ----------: |
| Samples         |            136 |         136 |
| R²              |         0.0067 |      0.0063 |
| Pseudo-F        |          0.910 |       0.852 |
| p-value         |          0.501 |       0.537 |

Among the 136 samples included in both analyses, gluten consumption explained less than 1% of the variation in fungal community composition (0.67% for whole ASVs and 0.63% for 16-mers). Both p-values were above 0.05.

### 3.2 Adjusted PERMANOVA

We put settlement type last in the formula to test whether fungal community composition differs between the gluten-consuming and gluten-avoiding sample groups after accounting for age and sex. Both tests use the same 136 samples as the unadjusted tests.

#### 3.2.1 Whole ASVs

In [30]:
!qiime diversity adonis \
    --i-distance-matrix "$asv_gluten_distance_path" \
    --m-metadata-file "$gluten_metadata_path" \
    --p-formula "age_years_sample + sex_sample + gluten_sample" \
    --o-visualization "$asv_adjusted_overall_path"

Saved Visualization to: ../data/processed/13_beta_diversity_gluten/adjusted_permanova/asv_overall.qzv


In [31]:
Visualization.load(asv_adjusted_overall_path)

<visualization: Visualization uuid: 4d228a00-e4a4-49f4-b7a4-9a7d3615cc44>

#### 3.2.2 16-mers

In [32]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_gluten_distance_path" \
    --m-metadata-file "$gluten_metadata_path" \
    --p-formula "age_years_sample + sex_sample + gluten_sample" \
    --o-visualization "$kmer16_adjusted_overall_path"

Saved Visualization to: ../data/processed/13_beta_diversity_gluten/adjusted_permanova/kmer16_overall.qzv


In [33]:
Visualization.load(kmer16_adjusted_overall_path)

<visualization: Visualization uuid: 7c23cd85-ecdf-49b0-aeff-3c0d72c68d42>

#### Adjusted results

| **Gluten term** | **Whole ASVs** | **16-mers** |
| --------------- | -------------: | ----------: |
| Samples         |            136 |         136 |
| R²              |         0.0053 |      0.0041 |
| Pseudo-F        |          0.719 |       0.561 |
| p-value         |          0.761 |       0.836 |

After accounting for age and sex, gluten consumption explained less than 1% of the variation in fungal community composition in both analyses (0.53% for whole ASVs and 0.41% for 16-mers). Both p-values were above 0.05.

### 4. Compare unadjusted and adjusted overall PERMANOVA

| **Bray–Curtis representation** | **Model**  | **Gluten R²** | **Pseudo-F** | **p-value** |
| ------------------------------ | ---------- | ------------: | -----------: | ----------: |
| Whole ASVs                     | Unadjusted |        0.0067 |        0.910 |       0.501 |
| Whole ASVs                     | Adjusted   |        0.0053 |        0.719 |       0.761 |
| 16-mers                        | Unadjusted |        0.0063 |        0.852 |       0.537 |
| 16-mers                        | Adjusted   |        0.0041 |        0.561 |       0.836 |

After adjustment, gluten R² decreased for both whole ASVs and 16-mers. Gluten consumption explained less than 1% of the variation in fungal community composition in all four tests, and all four p-values were above 0.05.